# Smart MCQ Solver Challenge

The objective of this project is to build models that can identify and rank the three most likely correct answers for multiple-choice questions.

## Overview

Each question in the dataset consists of a **prompt** and five answer choices: **A, B, C, D,** and **E**.

Instead of predicting only one answer, the model generates the **top three most likely answers in ranked order**. A higher rank for the correct answer results in a better evaluation score.

In this notebook, three different approaches are implemented and compared:

- A TF-IDF based retrieval approach
- A Logistic Regression classifier
- A Bi-LSTM neural network

All three models are evaluated using the same validation set.

## Evaluation

The competition uses **Mean Average Precision at 3 (MAP@3)** as the evaluation metric.

For a single question, the Average Precision at 3 (AP@3) is

$$
AP@3=
\begin{cases}
\frac{1}{r}, & \text{if the correct answer appears at rank } r \le 3,\\\\
0, & \text{otherwise.}
\end{cases}
$$

where $r$ is the position of the correct answer in the predicted top-3 list.

The final score is the mean of the AP@3 values over all questions.

$$
MAP@3=\frac{1}{N}\sum_{i=1}^{N} AP_i@3
$$

where $N$ is the total number of questions.

### Example

If the correct answer is **A**:

| Prediction | AP@3 |
|------------|------|
| `A B C` | 1.000 |
| `B A C` | 0.500 |
| `C D A` | 0.333 |
| `B C D` | 0.000 |

In addition to MAP@3, **Accuracy** and **Macro F1-score** are also reported using each model's top prediction.

## Dataset

| Column | Description |
|--------|-------------|
| `id` | Unique identifier for each question |
| `prompt` | Question or problem statement |
| `A`, `B`, `C`, `D`, `E` | Five answer choices |
| `answer` | Correct answer label (available only in the training data) |

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## 1. Imports & Setup

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

from collections import Counter
from typing import List, Dict

import wandb

import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

LABELS = ['A', 'B', 'C', 'D', 'E']

plt.style.use('ggplot')

## 2. Weights & Biases Setup

In [ ]:
# Weights & Biases setup 
!pip install -q wandb

import wandb

#wandb.login()
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

WANDB_PROJECT = "24f2001637-t22026"
print("W&B ready. Project:", WANDB_PROJECT)


## 3. Helper Functions

The following helper functions are used throughout the notebook to simplify preprocessing, evaluation, and submission generation.

- **`clean_text`**: Cleans and standardizes text by converting it to lowercase, removing extra whitespace, and filtering unwanted characters.
- **`label_to_idx` / `idx_to_label`**: Convert answer labels (`A` to `E`) to numerical indices (`0` to `4`) and vice versa.
- **`average_precision_at_k` / `map_at_3`**: Compute the Mean Average Precision at 3 (MAP@3) score used for evaluation.
- **`compute_metrics`**: Calculates Accuracy, Macro F1-score, and MAP@3 for model evaluation.
- **`format_submission`**: Creates the submission file in the required competition format.

In [ ]:
def clean_text(text: str) -> str:
    """Lowercase the text and remove extra/odd characters and whitespace."""
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)                       
    text = re.sub(r'[^\w\s.,!?;:()\-\'"]+', ' ', text) 
    return text.strip()


def build_option_text(prompt: str, option_text: str) -> str:
    """Join a question prompt with one answer option, separated by [SEP]."""
    return f"{clean_text(prompt)} [SEP] {clean_text(option_text)}"


def label_to_idx(label: str) -> int:
    """Turn a letter label ('A', 'B', ...) into an index (0, 1, ...)."""
    return ord(label.upper()) - ord('A')


def idx_to_label(idx: int) -> str:
    """Turn an index (0, 1, ...) back into a letter label ('A', 'B', ...)."""
    return chr(ord('A') + idx)


def average_precision_at_k(predicted: List[str], actual: str, k: int = 3) -> float:
    """Average precision for a single question: 1/rank if correct answer is in
    the top-k predictions, otherwise 0."""
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    rank = predicted.index(actual) + 1
    return 1.0 / rank


def map_at_3(predictions: List[List[str]], actuals: List[str]) -> float:
    """Mean Average Precision @ 3 across all questions (the competition metric)."""
    scores = [average_precision_at_k(pred, actual, k=3) for pred, actual in zip(predictions, actuals)]
    return float(np.mean(scores))


def compute_metrics(true_labels: List[str], top3_predictions: List[List[str]]) -> Dict[str, float]:
    """Compute the 3 common metrics used to compare runs:
    top-1 accuracy, macro F1 (both based on the single best prediction),
    and MAP@3 (based on the full top-3 ranking)."""
    top1_predictions = [pred[0] for pred in top3_predictions]
    return {
        "accuracy": accuracy_score(true_labels, top1_predictions),
        "f1_macro": f1_score(true_labels, top1_predictions, average="macro", labels=LABELS, zero_division=0),
        "map@3": map_at_3(top3_predictions, true_labels),
    }


def format_submission(ids, predictions, output_path: str) -> pd.DataFrame:
    """Save the top-3 predictions per question as a submission file."""
    submission = pd.DataFrame({
        'ID': ids,
        'Prediction': [' '.join(pred[:3]) for pred in predictions]
    })
    submission.to_csv(output_path, index=False)
    print(f"Saved submission to {output_path} ({len(submission)} rows)")
    return submission


## 4. Load the Data

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# Fill any missing option text with an empty string
for df in [train_df, test_df]:
    for col in LABELS:
        df[col] = df[col].fillna("").astype(str)

print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")
print("\nColumn types:")
print(train_df.dtypes)

train_df.head(3)

## 5. Explore the Data (EDA)

Before modeling, it helps to look at:
- how balanced the answer labels are
- how long the prompts and options are
- whether any prompts are duplicated

These checks catch data issues early and inform preprocessing decisions
in the next section.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

prompt_len = train_df["prompt"].str.split().str.len()
option_lens = {label: train_df[label].str.split().str.len() for label in LABELS}

# 1. How many questions have each correct answer letter
answer_counts = train_df["answer"].value_counts().sort_index()
axes[0, 0].bar(answer_counts.index, answer_counts.values)
axes[0, 0].set_title("Answer Label Distribution")
for i, v in enumerate(answer_counts.values):
    axes[0, 0].text(i, v, str(v), ha="center", va="bottom")

# 2. Prompt length in words
axes[0, 1].hist(prompt_len, bins=40)
axes[0, 1].axvline(prompt_len.mean(), linestyle="--")
axes[0, 1].set_title(f"Prompt Length (mean = {prompt_len.mean():.0f} words)")

# 3. Length of each option (A-E)
for label in LABELS:
    axes[0, 2].hist(option_lens[label], bins=30, alpha=0.5, label=label)
axes[0, 2].legend()
axes[0, 2].set_title("Option Lengths")

# 4. Prompt length split by correct answer
for label in LABELS:
    axes[1, 0].hist(prompt_len[train_df["answer"] == label], bins=20, alpha=0.5, label=label)
axes[1, 0].legend()
axes[1, 0].set_title("Prompt Length by Answer")

# 5. Total text length (prompt + all options)
total_len = prompt_len + sum(option_lens.values())
axes[1, 1].hist(total_len, bins=40)
axes[1, 1].set_title(f"Total Text Length (mean = {total_len.mean():.0f} words)")

# 6. How many prompts are exact duplicates
duplicate_count = train_df.duplicated("prompt", keep=False).sum()
axes[1, 2].bar(["Unique", "Duplicate"], [len(train_df) - duplicate_count, duplicate_count])
axes[1, 2].set_title("Prompt Uniqueness")

plt.tight_layout()
plt.show()

print(f"Duplicate prompts: {duplicate_count} / {len(train_df)}")


## 6. Preprocessing & Train/Validation Split

For each question, the prompt and all five answer choices are combined into a single text representation.

The training data is then split into training and validation sets using a 90:10 ratio. The validation set is used to evaluate all models under the same conditions before generating predictions on the test set.

In [ ]:
def build_mcq_text(row, include_opts=True):
    """Combine a question's prompt with its 5 options into one text block."""
    prompt = clean_text(str(row['prompt']))
    if not include_opts:
        return prompt
    options = " ".join([f"{label}: {clean_text(str(row[label]))}" for label in LABELS])
    return f"{prompt} [SEP] {options}"


train_sub, val_sub = train_test_split(train_df, test_size=0.1, random_state=42)
train_sub = train_sub.reset_index(drop=True)
val_sub = val_sub.reset_index(drop=True)

print(f"Train split: {len(train_sub)} | Validation split: {len(val_sub)}")

val_true = [str(row['answer']) for _, row in val_sub.iterrows()]

all_run_results = []


## 7. TF-IDF + Nearest Neighbours

A simple, **training-free** baseline that only relies on text similarity:

1. Turn every training question into a TF-IDF vector.
2. For a new question, find its most similar (cosine similarity) training questions.
3. Let those neighbours "vote" for the answer label, weighted by similarity.

It's fast to run and gives us a floor score that the other two models
should be able to beat.

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT,
    name="tfidf-knn-baseline",
    job_type="eval",
    config={"model": "tfidf-knn", "max_features": 50_000, "ngram_range": (1, 2), "k_neighbors": 15}
)

print("Fitting TF-IDF vectorizer...")

vectorizer = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

train_texts = train_sub.apply(build_mcq_text, axis=1)
train_embeddings = vectorizer.fit_transform(train_texts)

print(f"Vocabulary size: {len(vectorizer.vocabulary_):,} | Matrix shape: {train_embeddings.shape}")


def tfidf_predict(row, k=15):
    """Predict the top-3 answer labels using the k most similar training questions."""
    query = vectorizer.transform([build_mcq_text(row)])
    similarities = cosine_similarity(query, train_embeddings).ravel()
    neighbor_idx = np.argsort(similarities)[-k:][::-1]

    votes = {label: 0.0 for label in LABELS}
    for idx in neighbor_idx:
        score = similarities[idx]
        if score < 0.01:        
            continue
        neighbor_answer = train_sub.iloc[idx]["answer"].upper()
        if neighbor_answer in votes:
            votes[neighbor_answer] += score

    return sorted(votes, key=votes.get, reverse=True)[:3]


print("Predicting on the validation set...")
tfidf_val_preds = val_sub.apply(tfidf_predict, axis=1).tolist()

tfidf_metrics = compute_metrics(val_true, tfidf_val_preds)
print(f"Run 1 (TF-IDF + KNN) -> accuracy: {tfidf_metrics['accuracy']:.4f} | "
      f"F1 (macro): {tfidf_metrics['f1_macro']:.4f} | MAP@3: {tfidf_metrics['map@3']:.4f}")

wandb.log(tfidf_metrics)
all_run_results.append({"run": "tfidf-knn-baseline", **tfidf_metrics})
wandb.finish()


## 8. TF-IDF + Logistic Regression

This approach uses the same TF-IDF features created in the previous section. Instead of retrieving the most similar training example, a Logistic Regression classifier is trained using the training labels.

For each question, the model predicts the probability of each answer choice. These probabilities are then used to rank the top three predicted answers.

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT,
    name="tfidf-logistic-regression",
    job_type="eval",
    config={"model": "tfidf-logreg", "max_features": 50_000, "C": 1.0, "max_iter": 1000}
)

print("Training Logistic Regression classifier on TF-IDF features...")

logreg = LogisticRegression(max_iter=1000, C=1.0)
logreg.fit(train_embeddings, train_sub["answer"])

val_texts = val_sub.apply(build_mcq_text, axis=1)
val_embeddings = vectorizer.transform(val_texts)

val_probs = logreg.predict_proba(val_embeddings)
class_order = logreg.classes_

logreg_val_preds = [
    sorted(class_order, key=lambda label: -val_probs[i, list(class_order).index(label)])[:3]
    for i in range(val_probs.shape[0])
]

logreg_metrics = compute_metrics(val_true, logreg_val_preds)
print(f"Run 2 (TF-IDF + LogReg) -> accuracy: {logreg_metrics['accuracy']:.4f} | "
      f"F1 (macro): {logreg_metrics['f1_macro']:.4f} | MAP@3: {logreg_metrics['map@3']:.4f}")

wandb.log(logreg_metrics)
all_run_results.append({"run": "tfidf-logistic-regression", **logreg_metrics})
wandb.finish()

## 9. Bi-LSTM with Self-Attention

This model is trained from scratch using the training data.

For each question:

1. The prompt is paired with each of the five answer choices.
2. Each pair is encoded using a shared Bi-LSTM encoder.
3. A self-attention layer generates a fixed-length representation for each encoded sequence.
4. A scoring layer assigns a score to each answer choice.
5. The answer choice with the highest score is selected as the final prediction.

### 9.1 Build the Vocabulary

Before training the model, each word is converted into a numerical index. A vocabulary is created from the most frequent words in the training data, and these indices are used to represent the input text.

In [ ]:
all_training_texts = []
for _, row in train_sub.iterrows():
    prompt = clean_text(str(row['prompt']))
    for label in LABELS:
        all_training_texts.append(build_option_text(prompt, str(row[label])))

word_counts = Counter(word for text in all_training_texts for word in text.split())

VOCAB = {"<PAD>": 0, "<UNK>": 1}
for word, _ in word_counts.most_common(15000 - 2):
    VOCAB[word] = len(VOCAB)

VOCAB_SIZE = len(VOCAB)
MAX_LEN = 128
print(f"Vocabulary size: {VOCAB_SIZE:,}")


def encode(text, max_len=MAX_LEN):
    """Turn text into a fixed-length list of word ids (padded with 0s)."""
    ids = [VOCAB.get(word, 1) for word in text.lower().split()[:max_len]]
    return ids + [0] * (max_len - len(ids))

### 9.2 Dataset and Model Definition

- **`MCQDataset`**: Converts each question into five encoded prompt and answer choice pairs along with the corresponding target label.
- **`SelfAttn`**: Computes attention weights to identify the most informative words in each sequence and produces a fixed-length representation.
- **`BiLSTMModel`**: Encodes each prompt and answer choice pair using a shared Bi-LSTM and attention layer, then assigns a score to every answer choice.

In [ ]:
class MCQDataset(Dataset):
    """Wraps a dataframe of MCQ rows so PyTorch's DataLoader can batch them."""
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.has_answer = 'answer' in df.columns

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        prompt = clean_text(str(row['prompt']))
        options = torch.tensor(
            [encode(build_option_text(prompt, str(row[label]))) for label in LABELS],
            dtype=torch.long
        )
        label = torch.tensor(
            label_to_idx(str(row['answer'])) if self.has_answer else -1,
            dtype=torch.long
        )
        return options, label


class SelfAttn(nn.Module):
    """Learns an attention weight per timestep and returns a weighted sum
    of the Bi-LSTM outputs (a single summary vector per sequence)."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn_score = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x, mask=None):
        scores = self.attn_score(x).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1).unsqueeze(-1)
        return (x * weights).sum(dim=1)


class BiLSTMModel(nn.Module):
    """Shared Bi-LSTM + self-attention encoder, scores each of the 5 options."""
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            emb_dim, hidden_dim, num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.attention = SelfAttn(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def encode_option(self, x):
        """Encode one batch of (prompt + option) sequences into a single vector each."""
        mask = (x != 0)
        lstm_out, _ = self.lstm(self.dropout(self.embedding(x)))
        return self.dropout(self.attention(lstm_out, mask))

    def forward(self, options):
        """options shape: (batch, 5 options, seq_len) -> returns a score per option."""
        batch_size, num_options, seq_len = options.shape
        encoded = self.encode_option(options.view(batch_size * num_options, seq_len))
        return self.scorer(encoded).view(batch_size, num_options)


print("BiLSTM model defined.")

### 9.3 Train the Model

The model is trained using a standard PyTorch training loop. During each epoch, the training batches are processed through the model, the loss is computed, gradients are calculated using backpropagation, and the model parameters are updated with the optimizer.

After every epoch, the model is evaluated on the validation set using Accuracy, Macro F1-score, and MAP@3. These metrics are also logged to Weights & Biases (W&B) to monitor the training process.

In [ ]:
def run_training(model, train_df, val_df, epochs=12, batch_size=64, lr=3e-4, name="model"):
    train_loader = DataLoader(MCQDataset(train_df), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(MCQDataset(val_df), batch_size=batch_size, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    history = []
    best_map3 = 0.0
    final_val_preds = None

    for epoch in range(1, epochs + 1):

        # ----- Training -----
        model.train()
        total_loss, correct, samples = 0, 0, 0

        for options, labels in train_loader:
            options = options.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()
            logits = model(options)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            batch_n = labels.size(0)
            total_loss += loss.item() * batch_n
            correct += (logits.argmax(1) == labels).sum().item()
            samples += batch_n

        train_loss = total_loss / samples
        train_acc = correct / samples

        # ----- Validation -----
        model.eval()
        logits_list = []
        with torch.no_grad():
            for options, _ in val_loader:
                logits = model(options.to(DEVICE))
                logits_list.append(logits.cpu().numpy())

        logits = np.concatenate(logits_list)
        val_preds = [
            [idx_to_label(i) for i in np.argsort(row)[::-1][:3]]
            for row in logits
        ]
        final_val_preds = val_preds

        epoch_metrics = compute_metrics(val_true, val_preds)

        print(f"[{name}] E{epoch:02d} | TrainLoss={train_loss:.4f} | "
              f"TrainAcc={train_acc:.4f} | ValAcc={epoch_metrics['accuracy']:.4f} | "
              f"ValF1={epoch_metrics['f1_macro']:.4f} | ValMAP3={epoch_metrics['map@3']:.4f}")

        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_accuracy": epoch_metrics["accuracy"],
            "val_f1_macro": epoch_metrics["f1_macro"],
            "val_map@3": epoch_metrics["map@3"],
        })

        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc, **epoch_metrics})
        best_map3 = max(best_map3, epoch_metrics["map@3"])

    return history, best_map3, final_val_preds


run = wandb.init(
    project=WANDB_PROJECT,
    name="bilstm-self-attention",
    job_type="train",
    config={
        "model": "bilstm-attention", "vocab_size": VOCAB_SIZE, "max_len": MAX_LEN,
        "emb_dim": 128, "hidden_dim": 256, "num_layers": 2, "dropout": 0.3,
        "epochs": 12, "batch_size": 64, "lr": 3e-4
    }
)

bilstm = BiLSTMModel(VOCAB_SIZE).to(DEVICE)
num_params = sum(p.numel() for p in bilstm.parameters() if p.requires_grad)
print(f"BiLSTM parameter count: {num_params:,}")
wandb.config.update({"num_params": num_params})

bilstm_history, bilstm_best_map3, bilstm_val_preds = run_training(
    bilstm, train_sub, val_sub, epochs=12, name="bilstm"
)

bilstm_metrics = compute_metrics(val_true, bilstm_val_preds)
print(f"\nRun 3 (Bi-LSTM + Attention) final -> accuracy: {bilstm_metrics['accuracy']:.4f} | "
      f"F1 (macro): {bilstm_metrics['f1_macro']:.4f} | MAP@3: {bilstm_metrics['map@3']:.4f}")

wandb.summary.update(bilstm_metrics)
all_run_results.append({"run": "bilstm-self-attention", **bilstm_metrics})
wandb.finish()


## 10. Compare the 3 Runs

The performance of all three approaches is compared using the same validation set. Accuracy, Macro F1-score, and MAP@3 are used as the evaluation metrics for each model.

The metrics logged to Weights & Biases (W&B) can also be viewed and compared in the project dashboard. In addition, this notebook creates a local summary table using the recorded validation metrics.

In [ ]:
comparison_df = pd.DataFrame(all_run_results).set_index("run")
print(comparison_df)

comparison_df[["accuracy", "f1_macro", "map@3"]].plot(kind="bar", figsize=(9, 5))
plt.title("Model Comparison: Accuracy vs. Macro F1 vs. MAP@3")
plt.ylabel("Score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

best_run = comparison_df["map@3"].idxmax()
print(f"\nBest run by MAP@3: {best_run}")

## 11. Generate Predictions on the Test Set

The best-performing model, the Bi-LSTM, is used to generate predictions for the test set.

For each question, the model predicts the three most likely answer choices in ranked order. These predictions are then formatted according to the competition requirements and saved as `submission.csv`.

In [ ]:
print("Generating predictions on the test set using the Bi-LSTM model...")

def get_neural_probs(model, df, batch_size=64):
    """Run the model on a dataframe and return a softmax probability per label."""
    model.eval()
    loader = DataLoader(MCQDataset(df), batch_size=batch_size, shuffle=False)
    all_probs = []
    with torch.no_grad():
        for options, _ in loader:
            probs = torch.softmax(model(options.to(DEVICE)), dim=-1).cpu().numpy()
            all_probs.append(probs)
    all_probs = np.concatenate(all_probs)
    return [{label: float(all_probs[i, j]) for j, label in enumerate(LABELS)} for i in range(len(all_probs))]


test_probs = get_neural_probs(bilstm, test_df)

bilstm_predictions = [
    sorted(LABELS, key=lambda label: -test_probs[i][label])[:3]
    for i in range(len(test_df))
]

os.makedirs('../submissions', exist_ok=True)
id_col = 'id' if 'id' in test_df.columns else 'ID'

submission = format_submission(
    ids=test_df[id_col].tolist(),
    predictions=bilstm_predictions,
    output_path='submission.csv'
)

print("\nSample predictions:")
print(submission.head(10).to_string(index=False))
